In [ ]:
!pip install scikit-learn xgboost lightgbm catboost -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 10.3 MB/s eta 0:00:00


In [ ]:
import numpy as np               # массивы и математика
import pandas as pd              # таблицы
from sklearn.model_selection import train_test_split  # разделение данных
from sklearn.metrics import accuracy_score, mean_absolute_error  # метрики
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import (
    RandomForestClassifier,        # бэггинг со случайными признаками
    ExtraTreesClassifier,          # бэггинг с полной случайностью
    StackingClassifier,            # стекинг
    HistGradientBoostingClassifier # быстрый бустинг из sklearn
)
from xgboost import XGBClassifier   # градиентный бустинг от DMLC
from lightgbm import LGBMClassifier # градиентный бустинг от Microsoft
from catboost import CatBoostClassifier # градиентный бустинг от Яндекса

print("Библиотеки загружены!")

Библиотеки загружены!


In [ ]:
# ГЕНЕРАЦИЯ ДАННЫХ (клиенты интернет-магазина)
np.random.seed(42)                 # фиксируем генератор для воспроизводимости
N = 1000                           # количество клиентов

# Признаки (features)
visits = np.random.normal(5, 3, N).clip(0)    # визитов в месяц (нормальное распределение)
avg_order = np.random.normal(3000, 1500, N).clip(0) # средний чек, руб.
days_since = np.random.exponential(30, N)      # дней с последнего визита
returns_ratio = np.random.beta(2, 10, N)       # доля возвратов (от 0 до 1)

# Задача регрессии: предсказываем среднемесячную выручку
revenue = visits * avg_order * (1 - returns_ratio) + np.random.normal(0, 2000, N)
y_reg = revenue

# Задача классификации: предсказываем отток (1 — ушёл, 0 — остался)
# Для создания дисбаланса классов и усложнения задачи используем сигмоидную функцию
churn_prob = 1 / (1 + np.exp(-( -1.5 + returns_ratio*3 - visits*0.2 + days_since*0.05 )))
churn = (np.random.random(N) < churn_prob).astype(int)
y_clf = churn

# Объединяем признаки в одну матрицу X
X = np.column_stack([visits, avg_order, days_since, returns_ratio])

# Разделяем данные на тренировочную и тестовую выборки для классификации
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X, y_clf, test_size=0.2, random_state=42)
# Разделяем данные для регрессии
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X, y_reg, test_size=0.2, random_state=42)

print("Данные готовы!")

Данные готовы!


In [ ]:
# СРАВНЕНИЕ МОДЕЛЕЙ ДЛЯ КЛАССИФИКАЦИИ
# Создаём словарь с моделями: ключ — название, значение — объект модели
models = {
    # Одиночная модель (baseline)
    'Logistic Regression': LogisticRegression(max_iter=500),

    # Бэггинг
    'Random Forest': RandomForestClassifier(n_estimators=100),  # 100 деревьев
    'Extra Trees': ExtraTreesClassifier(n_estimators=100),      # ещё больше случайности

    # Бустинг
    'HistGradientBoosting': HistGradientBoostingClassifier(),   # sklearn-бустинг
    'XGBoost': XGBClassifier(eval_metric='logloss'),            # требуется отключить вывод метрики
    'LightGBM': LGBMClassifier(verbose=-1),                     # отключаем логирование
    'CatBoost': CatBoostClassifier(verbose=0)                   # отключаем вывод прогресса
}

print("Accuracy (доля правильных ответов) на задаче оттока клиентов:\n")
# Проходим по всем моделям в словаре
for name, model in models.items():
    # fit — обучаем модель на тренировочных данных
    model.fit(X_train_clf, y_train_clf)
    # predict — предсказываем классы для тестовых данных
    pred = model.predict(X_test_clf)
    # accuracy_score сравнивает предсказания с реальными метками
    acc = accuracy_score(y_test_clf, pred)
    print(f"{name:<25}: {acc:.3f}")

Accuracy (доля правильных ответов) на задаче оттока клиентов:

Logistic Regression      : 0.750
Random Forest            : 0.695
Extra Trees              : 0.695
HistGradientBoosting     : 0.660
XGBoost                  : 0.655


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


LightGBM                 : 0.670
CatBoost                 : 0.715


In [ ]:
# ДЕМОНСТРАЦИЯ СТЕКИНГА
# Базовые модели (могут быть любыми)
base_models = [
    ('rf', RandomForestClassifier(n_estimators=100)),           # первый эксперт
    ('hgb', HistGradientBoostingClassifier())                  # второй эксперт
]
# Мета-ученик (учится на предсказаниях экспертов)
stack = StackingClassifier(estimators=base_models, final_estimator=LogisticRegression())
stack.fit(X_train_clf, y_train_clf)
acc_stack = accuracy_score(y_test_clf, stack.predict(X_test_clf))
print(f"\nСтекинг (RF + HGB + LogReg): {acc_stack:.3f}")


Стекинг (RF + HGB + LogReg): 0.695


In [ ]:
# БЫСТРЫЙ ПРИМЕР ДЛЯ РЕГРЕССИИ (предсказание выручки)
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor

reg_models = {
    'Linear Regression': LinearRegression(),
    'Random Forest Regressor': RandomForestRegressor(n_estimators=100),
    'HistGradientBoosting Regressor': HistGradientBoostingRegressor()
}

print("\nСредняя абсолютная ошибка (MAE) на задаче регрессии:\n")
for name, model in reg_models.items():
    model.fit(X_train_reg, y_train_reg)
    pred = model.predict(X_test_reg)
    mae = mean_absolute_error(y_test_reg, pred)
    print(f"{name:<35}: {mae:.0f} руб.")


Средняя абсолютная ошибка (MAE) на задаче регрессии:

Linear Regression                  : 3091 руб.
Random Forest Regressor            : 1938 руб.
HistGradientBoosting Regressor     : 2005 руб.


## 🎸 ТВОРЧЕСКОЕ ЗАДАНИЕ

1. Измени количество деревьев в Random Forest (n_estimators=10, 500). Как меняется точность и время обучения?
2. Попробуй добавить в стекинг больше моделей (например, Extra Trees или XGBoost).
3. Для регрессии сравни Random Forest и градиентный бустинг. Какой дает меньшую ошибку?
4. Самостоятельно реализуй простой блендинг: раздели train на train_base и val, обучи базовые модели на train_base, получи их предсказания на val и обучи мета-модель на этих предсказаниях.